<a href="https://colab.research.google.com/github/aditya-nannaware-rex/SQL_AUTOMATED_AGENT/blob/main/SQL_AUTOMATED_AGENT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sqlite3
import time

def setup_database():
    conn = sqlite3.connect('employee_database.db')
    cursor = conn.cursor()

    # Create Tables
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS Department (
            id INTEGER PRIMARY KEY,
            name TEXT
        )
    ''')
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS Employee (
            id INTEGER PRIMARY KEY,
            name TEXT,
            salary INTEGER,
            department_id INTEGER,
            FOREIGN KEY (department_id) REFERENCES Department(id)
        )
    ''')

    # Clear existing data to avoid duplicates if you run this cell twice
    cursor.execute('DELETE FROM Employee')
    cursor.execute('DELETE FROM Department')

    # Insert Dummy Data
    cursor.executemany('INSERT INTO Department (id, name) VALUES (?, ?)', [
        (1, 'Sales'), (2, 'HR'), (3, 'Engineering'), (4, 'Marketing')
    ])

    cursor.executemany('INSERT INTO Employee (id, name, salary, department_id) VALUES (?, ?, ?, ?)', [
        (101, 'Alice', 85000, 1), (102, 'Bob', 90000, 1),
        (103, 'Charlie', 60000, 2), (104, 'David', 65000, 2),
        (105, 'Eve', 120000, 3), (106, 'Frank', 115000, 3), (107, 'Grace', 130000, 3),
        (108, 'Heidi', 70000, 4)
    ])

    conn.commit()
    conn.close()
    print("Database 'employee_database.db' created and populated successfully.")

setup_database()

In [ ]:
import os
import sqlite3
from google import genai
from google.genai import types
from google.colab import userdata

# --- 1. GET API KEY ---
try:
    api_key = userdata.get('GEMINI_API_KEY')
    os.environ["GEMINI_API_KEY"] = api_key
except userdata.SecretNotFoundError:
    print("Error: Please add 'GEMINI_API_KEY' to your Colab Secrets.")

# --- 2. AUTOMATED SCHEMA EXTRACTOR ---
def get_dynamic_schema(db_path='employee_database.db') -> str:
    """Reads the database and returns a string of all tables and columns."""
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Query SQLite's internal master table to get all CREATE TABLE statements
    cursor.execute("SELECT sql FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'")
    tables = cursor.fetchall()
    conn.close()

    schema_string = ""
    for table in tables:
        # table[0] contains the raw "CREATE TABLE..." string
        schema_string += f"{table[0]}\n"

    return schema_string

# --- 3. DEFINE THE TOOL ---
def execute_sql(query: str) -> str:
    """Executes a SQL query and returns the formatted results or an error message."""
    import time
    time.sleep(15) # Rate limit hack for free tier
    try:
        conn = sqlite3.connect('employee_database.db')
        cursor = conn.cursor()
        cursor.execute(query)

        if query.strip().upper().startswith("SELECT"):
            columns = [description[0] for description in cursor.description]
            rows = cursor.fetchall()
            if not rows: return "Query successful. 0 rows returned."

            result_string = f"Columns: {', '.join(columns)}\n"
            for row in rows[:10]: result_string += str(row) + "\n"
            if len(rows) > 10: result_string += f"\n...and {len(rows) - 10} more rows."
            return result_string
        else:
            conn.commit()
            return "Query executed successfully. (Data modified)"
    except sqlite3.Error as e:
        return f"SQL Error: {e}"
    finally:
        if conn: conn.close()

# --- 4. CONFIGURE THE AGENT WITH DYNAMIC INJECTION ---

# Fetch the schema dynamically right now
current_db_schema = get_dynamic_schema()

# Inject it into the prompt using an f-string
system_instruction = f"""
You are an expert SQL Data Analyst interacting with a local SQLite database.
Here is the exact structure of the database you are working with:

{current_db_schema}

Rules:
1. ALWAYS use the `execute_sql` tool to answer data questions.
2. ALWAYS use explicit table aliases (e.g., TableName.column) to avoid ambiguity in joins.
3. Output your final response in two parts:
   - Part 1: The plain-English answer based STRICTLY on the data returned. Do not guess.
   - Part 2: The exact SQL query you used to get that data.
"""

client = genai.Client()
config = types.GenerateContentConfig(
    system_instruction=system_instruction,
    tools=[execute_sql],
    temperature=0.0
)

# Initialize the chat session
chat = client.chats.create(model="gemini-2.5-flash", config=config)
print("Agent initialized with dynamic schema. Ready to query.")

In [ ]:
user_question = "What is the name of the department with the highest average salary, and what is that average?"

print(f"User: {user_question}\n")
response = chat.send_message(user_question)
print(f"Agent:\n{response.text}")

In [ ]:
import sqlite3

def setup_advanced_database():
    conn = sqlite3.connect('employee_database.db')
    cursor = conn.cursor()

    # 1. Clear old tables
    tables = ['Employee', 'Department', 'Users', 'Trips', 'Projects', 'EmployeeProject']
    for table in tables:
        cursor.execute(f'DROP TABLE IF EXISTS {table}')

    # 2. Create Tables
    cursor.executescript('''
        CREATE TABLE Department (id INTEGER PRIMARY KEY, name TEXT);
        CREATE TABLE Employee (id INTEGER PRIMARY KEY, name TEXT, salary INTEGER, department_id INTEGER);

        -- LeetCode 'Trips and Users' Schema
        CREATE TABLE Users (users_id INTEGER PRIMARY KEY, banned TEXT, role TEXT);
        CREATE TABLE Trips (id INTEGER PRIMARY KEY, client_id INTEGER, driver_id INTEGER, city_id INTEGER, status TEXT, request_at TEXT);

        -- Projects Schema
        CREATE TABLE Projects (project_id INTEGER PRIMARY KEY, name TEXT, budget INTEGER);
        CREATE TABLE EmployeeProject (employee_id INTEGER, project_id INTEGER, hours_worked INTEGER);
    ''')

    # 3. Insert Data
    cursor.executemany('INSERT INTO Department VALUES (?, ?)', [
        (1, 'IT'), (2, 'Sales'), (3, 'Executive')
    ])

    cursor.executemany('INSERT INTO Employee VALUES (?, ?, ?, ?)', [
        (1, 'Joe', 85000, 1), (2, 'Henry', 80000, 2), (3, 'Sam', 60000, 2),
        (4, 'Max', 90000, 1), (5, 'Janet', 69000, 1), (6, 'Randy', 85000, 1),
        (7, 'Will', 70000, 1)
    ])

    cursor.executemany('INSERT INTO Users VALUES (?, ?, ?)', [
        (1, 'No', 'client'), (2, 'Yes', 'client'), (3, 'No', 'client'),
        (4, 'No', 'client'), (10, 'No', 'driver'), (11, 'No', 'driver'),
        (12, 'No', 'driver'), (13, 'No', 'driver')
    ])

    cursor.executemany('INSERT INTO Trips VALUES (?, ?, ?, ?, ?, ?)', [
        (1, 1, 10, 1, 'completed', '2023-10-01'), (2, 2, 11, 1, 'cancelled_by_driver', '2023-10-01'),
        (3, 3, 12, 6, 'completed', '2023-10-01'), (4, 4, 13, 6, 'cancelled_by_client', '2023-10-01'),
        (5, 1, 10, 1, 'completed', '2023-10-02'), (6, 2, 11, 6, 'completed', '2023-10-02'),
        (7, 3, 12, 6, 'completed', '2023-10-02'), (8, 2, 12, 12, 'completed', '2023-10-03'),
        (9, 3, 10, 12, 'completed', '2023-10-03'), (10, 4, 13, 12, 'cancelled_by_driver', '2023-10-03')
    ])

    cursor.executemany('INSERT INTO Projects VALUES (?, ?, ?)', [
        (101, 'AI Migration', 150000), (102, 'Data Warehouse', 200000), (103, 'Frontend Revamp', 50000)
    ])

    cursor.executemany('INSERT INTO EmployeeProject VALUES (?, ?, ?)', [
        (1, 101, 40), (1, 102, 20), (2, 103, 50), (4, 101, 35), (5, 101, 10), (7, 102, 60)
    ])

    conn.commit()
    conn.close()
    print("Advanced Schema Database created and populated successfully.")

setup_advanced_database()

In [ ]:
user_question = "Calculate the cancellation rate for requests made by unbanned clients and driven by unbanned drivers, grouped by day. The cancellation rate is the number of canceled requests (by client or driver) divided by the total number of requests that day. Round the rate to two decimal places."

print(f"User: {user_question}\n")
response = chat.send_message(user_question)
print(f"Agent:\n{response.text}")